## Profile a training job to resolve GPU utilization bottlenecks
The goal of this lab is to train the model and generate a nsight profile report. Compare the generated report with the report shared with you to identify the improvements to the **gpu utilization**. You may use the solution provided in this notebook or create your own notebook to try your own solution as well. Also, feel free to tweak the configurations use in the below sample solution.

We will execute all cells in this notebook normally, and only profile the last cell `lora_trainer.train()`. For the last cell, we will use Nsight Jupyterlab extension to execute and profile. Once the report is generated, open the Nsight streamer to analyze the report and observe any memory bottlenecks.
Fix as many bottlenecks as you can, and keep reviewing your changes by regenerating the reports. 

**Pre-requisites:**
- Stop any existing active Kernels, and restart this Kernel using the 'Restart Kernel' button in the toolbar.
- Enable 'Profiling with Nsight Systems' from 'NVIDIA Nsight' extension in the toolbar
- Select 'Restart Kernel' when prompted, and wait until you see a **Green** Run icon in the toolbar. Use this icon to run the last cell (`lora_trainer,train()`).

In [ ]:
!pip install -U datasets transformers accelerate trl

In [ ]:
import torch
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, IntervalStrategy
from trl import SFTTrainer

dataset = "greenwich157/telco-5G-core-faults"
model_id= "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

def prepare_training_data(dataset_id):
    data = load_dataset(dataset_id, split="train")
    data_df = data.to_pandas()
    data_df["text"] = data_df[["instruction", "input", "output"]].apply(lambda x:"<|system|>\n" + x.instruction + "</s><|user|>\n" + x.input + "</s>\n<|assistant|>\n" + x.output + "</s>", axis=1)
    data = Dataset.from_pandas(data_df)
    return data

full_train_data = prepare_training_data(dataset)
print ("training data prepared successfully ", full_train_data)
full_train_data = full_train_data.remove_columns(['instruction', 'input', 'output'])

full_train_data = full_train_data.select(range(180)) # trimming to reduce report size
full_train_data = full_train_data.train_test_split(test_size=0.20, seed=42)
print ("training data trimmed and split successfully ", full_train_data)
train_data = full_train_data['train']
test_data = full_train_data['test']

torch.cuda.empty_cache()

In [ ]:
ft_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="cuda",
    dtype=torch.float32
)

ft_train_args = TrainingArguments(
    batch_eval_metrics=True,
    disable_tqdm=False,
    dataloader_num_workers=4,
    dataloader_persistent_workers=True,
    dataloader_drop_last=True,
    dataloader_pin_memory=False,
    dataloader_prefetch_factor=2,
    do_eval=True,
    eval_accumulation_steps=4,
    eval_do_concat_batches=False,
    eval_strategy=IntervalStrategy.STEPS,
    full_determinism=True, 
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    logging_strategy=IntervalStrategy.STEPS,
    logging_steps=6,
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    max_grad_norm=1.0,
    num_train_epochs=2,
    optim="adamw_torch",
    output_dir="ft-tinyllama-fp32-singlegpu",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=4,
    prediction_loss_only=False,
    remove_unused_columns=False,
    skip_memory_metrics=False,
    save_strategy=IntervalStrategy.STEPS,
    save_steps=50,
    warmup_steps=0,
    warmup_ratio=0.1,
)
print ("training arguments initialized successfully ", ft_train_args)

ft_trainer = SFTTrainer(
    model=ft_model,
    args=ft_train_args,
    train_dataset=train_data, 
    eval_dataset=test_data,
)
print ("Trainer class initialized and staring full fine-tuning")

In [ ]:
torch.cuda.device_count()

In [ ]:
print(f"GPU 0 memory: {torch.cuda.get_device_properties(0).total_memory}")
print(f"GPU 0 allocated: {torch.cuda.memory_allocated(0)}")

In [ ]:
print(ft_model.hf_device_map)

In [ ]:
ft_trainer.train()
print ("full fine-tuning completed successfully")

## **Sample Profiling Report**

![](../../static-assets/lab-02/01-gpu-utilization-baseline-solution-rep1.png)

![](../../static-assets/lab-02/01-gpu-utilization-baseline-solution-rep2.png)

![](../../static-assets/lab-02/01-gpu-utilization-baseline-solution-rep3.png)